# 1. Download Lidar and DEM

In [1]:
import requests
import xml.etree.ElementTree as ET
import geopandas as gpd
from shapely.geometry import Polygon
import os
import re
import pandas as pd

In [3]:
os.getcwd()

'c:\\Users\\Hojung Yu\\Documents\\GitHub\\fast-thermal-comfort\\01_Lidar_to_DEM'

In [45]:
### --- Configuration (User MUST update these paths) --- ###
TIFF_URL_LIST_FILE = r"C:\Users\Hojung Yu\Desktop\lidar_download\miami_dem.txt" # This will be downloaded text file.
LIDAR_URL_LIST_FILE = r"C:\Users\Hojung Yu\Desktop\lidar_download\miami_lidar.txt" # DEM directory
DEST_DIR_DEM = r"C:\Users\Hojung Yu\Desktop\lidar_download\miami" # DEM directory
DEST_DIR_LIDAR = r"C:\Users\Hojung Yu\Desktop\lidar_download\miami" # LiDAR directory 
BOUNDARY_FILE = r"C:\Users\Hojung Yu\Desktop\lidar_download\miami_2.geojson" # Atlanta Boundary geojson. Used buffered version to make sure you have all tiles inside.
### ---------------------------------------------------- ###

In [31]:
def load_boundary(boundary_filepath):
    """
    Loads city boundary using geopandas.
    Args:
        boundary_filepath (str): Path to the geospatial file for Atlanta's boundary.
    Returns:
        gpd.GeoDataFrame: GeoDataFrame containing Atlanta's boundary.
    """
    try:
        boundary_gdf = gpd.read_file(boundary_filepath)
        if boundary_gdf.crs is None or boundary_gdf.crs.to_epsg() != 4326:
            print(f"Warning: Atlanta boundary CRS is {boundary_gdf.crs}. Reprojecting to EPSG:4326.")
            boundary_gdf = boundary_gdf.to_crs(epsg=4326)
        print(f"Successfully loaded Atlanta boundary from: {boundary_filepath}")
        return boundary_gdf
    except Exception as e:
        print(f"Could not load Atlanta boundary from '{boundary_filepath}': {e}")
        print("Using a simplified placeholder bounding box for Atlanta. Please provide a real file for accuracy.")
        # west = -84.50
        # south = 33.65
        # east = -84.30
        # north = 33.85
        # atlanta_bbox_polygon = Polygon([(west, south), (east, south), (east, north), (west, north), (west, south)])
        # return gpd.GeoDataFrame(
        #     {'geometry': [atlanta_bbox_polygon]},
        #     crs="EPSG:4326"
        # )
        return print("Using a simplified placeholder bounding box for Atlanta. Please provide a real file for accuracy.")

def construct_xml_url(file_url):
    """
    Convert a USGS LAZ/TIFF URL into its corresponding metadata XML URL.
    
    Steps:
    1. Replace '/LAZ/' or '/TIFF/' with '/metadata/'.
    2. Replace the extension (.laz or .tif) with .xml.
    """
    # Step 1: Replace folder
    if "/LAZ/" in file_url:
        xml_url = file_url.replace("/LAZ/", "/metadata/")
    elif "/TIFF/" in file_url:
        xml_url = file_url.replace("/TIFF/", "/metadata/")
    else:
        raise ValueError("URL does not contain /LAZ/ or /TIFF/")

    # Step 2: Replace extension
    if xml_url.endswith(".laz"):
        xml_url = xml_url[:-4] + ".xml"
    elif xml_url.endswith(".tif"):
        xml_url = xml_url[:-4] + ".xml"
    else:
        raise ValueError("URL does not end with .laz or .tif")

    return xml_url

def download_file(url, destination_folder):
    """
    Downloads a file from a given URL to a specified folder.
    Returns the path to the downloaded file, or None on failure.
    """
    # os.makedirs(destination_folder, exist_ok=True)
    local_filename = os.path.join(destination_folder, url.split('/')[-1])
    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(local_filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        # print(f"Downloaded: {local_filename}")
        return local_filename
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {url}: {e}")
        return None


def parse_xml_bounding_box(xml_filepath):
    """
    Parses an XML file to extract bounding box coordinates.
    Returns a shapely Polygon representing the bounding box, or None if not found/parsed.
    """
    try:
        tree = ET.parse(xml_filepath)
        root = tree.getroot()

        # Define the namespace if present, or search without it

        def find_element_text(parent, tag_name):
            # Search for element with or without namespace
            element = parent.find(tag_name)
            if element is None:
                # Try with common namespaces
                for ns_prefix in ['', '{http://www.fgdc.gov/metadata/fgdc-std-001-1998.xsd}', '{http://www.isotc211.org/2005/gmd}']:
                    element = parent.find(f'{ns_prefix}{tag_name}')
                    if element is not None:
                        break
            return element.text if element is not None else None

        # Find the bounding box elements
        westbc = find_element_text(root.find('.//bounding'), 'westbc')
        eastbc = find_element_text(root.find('.//bounding'), 'eastbc')
        northbc = find_element_text(root.find('.//bounding'), 'northbc')
        southbc = find_element_text(root.find('.//bounding'), 'southbc')

        if all([westbc, eastbc, northbc, southbc]):
            west = float(westbc)
            east = float(eastbc)
            north = float(northbc)
            south = float(southbc)
            bbox_polygon = Polygon([(west, south), (east, south), (east, north), (west, north), (west, south)])
            return bbox_polygon
        else:
            print(f"Warning: Could not find all bounding box coordinates in {xml_filepath}")
            return None
    except ET.ParseError as e:
        print(f"Error parsing XML file {xml_filepath}: {e}")
        return None
    except ValueError as e:
        print(f"Error converting bounding box coordinates to float in {xml_filepath}: {e}")
        return None
    except AttributeError: # Happens if .//bounding is not found
        print(f"Warning: 'bounding' element not found in XML file {xml_filepath}.")
        return None

In [32]:
def download_lidar_or_tiff(URL_LIST_TEXT, BOUNDARY_GDF, DEST_DIR):
    # 2. Process the list of URLs
    try:
        with open(URL_LIST_TEXT, 'r') as f:
            urls = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        print(f"Error: TIFF URL list file not found at {URL_LIST_TEXT}")
        return

    print(f"\nProcessing {len(urls)} TIFF URLs...")

    # 3. Process the lists of TIFFs
    for i, tif_url in enumerate(urls):
        xml_url = construct_xml_url(tif_url)
        if not xml_url:
            print(f"Skipping: Could not construct XML URL for {tif_url}")
            continue

        xml_filepath = download_file(xml_url, DOWNLOAD_DIR_LIDAR)
        if not xml_filepath:
            print(f"Skipping: Failed to download XML for {tif_url}")
            continue
        data_bbox_polygon = parse_xml_bounding_box(xml_filepath)
        try:
            os.remove(xml_filepath)
            # print(f"Removed temporary XML file: {xml_filepath}")
        except OSError as e:
            print(f"Error removing temporary XML file {xml_filepath}: {e}")
        if data_bbox_polygon:
            data_bbox_geoseries = gpd.GeoSeries([data_bbox_polygon], crs="EPSG:4326")
            if data_bbox_geoseries.intersects(BOUNDARY_GDF).any():

                print(f"Initiating download for TIFF: {tif_url}")
                download_file(tif_url, DEST_DIR)
        else:
            print(f"Skipping: Could not get valid bounding box for ID {tif_url} from XML.")

In [48]:
def main():
    # 1. Load Atlanta's boundary
    boundary_gdf = load_boundary(BOUNDARY_FILE)
    if boundary_gdf is None or boundary_gdf.empty:
        print("Failed to load or create boundary. Exiting.")
        return

    boundary_gdf = boundary_gdf.geometry.unary_union
    print()

    os.makedirs(DEST_DIR_DEM, exist_ok=True)
    os.makedirs(DEST_DIR_LIDAR, exist_ok=True)
    print("process: downloading TIFF (DEM)")
    # download_lidar_or_tiff(TIFF_URL_LIST_FILE, boundary_gdf, DEST_DIR_DEM)
    print("process: downloading LiDAR")
    download_lidar_or_tiff(LIDAR_URL_LIST_FILE, boundary_gdf, DEST_DIR_LIDAR)

In [49]:
if __name__== "__main__":
    main()

Successfully loaded Atlanta boundary from: C:\Users\Hojung Yu\Desktop\lidar_download\miami_2.geojson

process: downloading TIFF (DEM)
process: downloading LiDAR

Processing 1534 TIFF URLs...


C:\Users\HojungYu\AppData\Local\Temp\ipykernel_25220\4051600386.py:8: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  boundary_gdf = boundary_gdf.geometry.unary_union


Initiating download for TIFF: https://rockyweb.usgs.gov/vdelivery/Datasets/Staged/Elevation/LPC/Projects/FL_MiamiDade_D23/FL_MiamiDade_1_D23/LAZ/USGS_LPC_FL_MiamiDade_D23_LID2024_317552_0901.laz
Initiating download for TIFF: https://rockyweb.usgs.gov/vdelivery/Datasets/Staged/Elevation/LPC/Projects/FL_MiamiDade_D23/FL_MiamiDade_1_D23/LAZ/USGS_LPC_FL_MiamiDade_D23_LID2024_317849_0901.laz


KeyboardInterrupt: 